In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pydol.photometry.scripts.cmdtools import gen_CMD
import subprocess
from astropy.table import Table
from astropy.modeling import models
import glob
from astropy.coordinates import angular_separation
import astropy.units as u
from astropy.modeling import models, fitting
from pydol.photometry.scripts.catalog_filter import *
from matplotlib.ticker import (MultipleLocator, AutoLocator, AutoMinorLocator)
import seaborn as sb
import matplotlib as mpl

In [ ]:
Av_dict = { # WFC3
            'f275w': 2.02499,
            'f336w': 1.67536,
            'f438w': 1.34148,
            'f606w': 0.90941,
            'uf814w': 0.59845,
            
            # JWST-NIRCAM
            'f090w': 0.583,
            'f115w': 0.419,
            'f140m': 0.315,
            'f150w': 0.287,
            'f200w': 0.195,
            'f212n': 0.176,
            'f356w': 0.099,
            'f444w': 0.083,
            # WFC2
            'f435w': 1.33879,
            'f555w': 1.03065,
            'f814w': 0.59696,
          }

In [ ]:
# Minimalistic seaborn style
sb.set_theme(style="white")

mpl.rcParams.update({
    "text.usetex": True,                # If using LaTeX for labels
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "axes.labelsize": 25,
    "font.size": 25,
    "legend.fontsize": 10,
    "xtick.labelsize": 30,
    "ytick.labelsize": 30,
    "axes.titlesize": 30,
    "lines.linewidth": 1.0,
    "lines.markersize": 3,
    "figure.dpi": 300,                  # High-quality output
    "savefig.dpi": 300,
    "axes.grid": False,                 # Avoid grids unless needed
    "legend.frameon": False             # No legend frame
})

plt.rcParams['axes.titlesize'] = plt.rcParams['axes.labelsize'] = 30
plt.rcParams['xtick.labelsize'] = plt.rcParams['ytick.labelsize'] = 30


In [ ]:
def gen_CMD(
    tab,
    df_iso = None,
    filters={'filt1': 'f115w', 'filt2': 'f150w'},
    positions={'ra_col': 'ra', 'dec_col': 'dec', 'ra_cen': 0, 'dec_cen': 0},
    region={'r_in': 0, 'r_out': 24, 'spatial_filter': 'circle','ang': 245.00492},
    extinction={'Av': 0.19, 'Av_': 3, 'Av_x': 2, 'Av_y': 19},
    distance_modulus=29.7415,
    axis_limits={'xlims': [-0.5, 2.5], 'ylims': [18, 28]},
    isochrone_params={'met': 0.02, 'ages': [7., 8., 9.]},
    plot_settings={'alpha': 1, 's': 0.2, 'lw': 3},
    error_settings={ 'mag_err_lim': 0.2, 'show_err_model': False, 'ref_xpos': -0.5},
    kde_contours={'gen_kde': False, 'gen_contours': False},
    other_settings={'ab_dist': True, 'skip_data': False, 'show_err_model':False},
    fig=None,
    ax=None):
    
    """
    Generate a Color-Magnitude Diagram (CMD) with optional KDE or contour overlays.

    Parameters
    ----------
    tab : DataFrame
        Input data table containing magnitudes, positions, and errors for sources.
        
    df_iso : DataFrame, optional
        Isochrone data for overlay.

    filters : dict, optional
        Filters used in CMD. Keys:
        - 'filt1': Primary filter for color calculation.
        - 'filt2': Secondary filter for color calculation.
        - 'filt3': Filter for magnitude axis. Defaults to 'filt2'.

    positions : dict, optional
        Positional parameters. Keys:
        - 'ra_col': RA column name.
        - 'dec_col': DEC column name.
        - 'ra_cen': Central RA (in degrees).
        - 'dec_cen': Central DEC (in degrees).

    region : dict, optional
        Region parameters for filtering sources. Keys:
        - 'r_in': Inner radius for selection (arcseconds) (used for circular filters).
        - 'r_out': Outer radius for selection (arcseconds) (used for circular filters).
        - 'spatial_filter': Type of spatial filtering ('circle', 'box', 'ellipse').
        - 'ang': Orientation angle (used for box or ellipse filters).
        - 'width_in', 'height_in': Inner box dimensions (arcseconds) (used for box filters).
        - 'width_out', 'height_out': Outer box dimensions (arcseconds) (used for box filters).
        - 'a1', 'b1': Inner semi-major and semi-minor axes (arcseconds) (used for ellipse filters).
        - 'a2', 'b2': Outer semi-major and semi-minor axes (arcseconds) (used for ellipse filters).

    extinction : dict, optional
        Extinction parameters. Keys:
        - 'Av': Extinction value.
        - 'Av_': Annotation extinction value.
        - 'Av_x', 'Av_y': Annotation arrow position.

    distance_modulus : float, optional
        Distance modulus for CMD adjustments. Default is 29.7415.

    axis_limits : dict, optional
        Plot axis limits. Keys:
        - 'xlims': Limits for x-axis (color).
        - 'ylims': Limits for y-axis (magnitude).

    isochrone_params : dict, optional
        Isochrone parameters for plotting. Keys:
        - 'met': Metallicity for isochrones.
        - 'label_min': Minimum label value for filtering.
        - 'label_max': Maximum label value for filtering.
        - 'ages': List of log ages to plot.

    plot_settings : dict, optional
        General plot settings. Keys:
        - 'alpha': Transparency for isochrone lines.
        - 's': Marker size for scatter plots.
        - 'lw': Line width for isochrones.

    error_settings : dict, optional
        Settings for error handling and plotting. Keys:
        - 'mag_err_cols': Columns for magnitude errors. Defaults to filter-based columns.
        - 'mag_err_lim': Maximum allowable magnitude error.
        - 'show_err_model': Show error models during plotting.
        - 'ref_xpos': Reference x-position for error bars.

    kde_contours : dict, optional
        Settings for KDE or contour plots. Keys:
        - 'gen_kde': Generate KDE overlay.
        - 'gen_contours': Generate contour overlay.

    other_settings : dict, optional
        Miscellaneous settings. Keys:
        - 'ab_dist': Include absolute distance modulus adjustments.
        - 'skip_data': Skip scatter plot of source data.

    fig : matplotlib.figure.Figure, optional
        Existing figure object. If None, a new figure is created.

    ax : matplotlib.axes.Axes, optional
        Existing axis object. If None, a new axis is created.


    Returns
    -------
    tuple
        (fig, ax, tab) where:
        - fig: The figure object.
        - ax: The axis object.
        - tab: The filtered input data table after spatial and error-based selection.
    """
    
    # Fill in default values for nested dictionaries
    filters.setdefault('filt1','f115w')
    filters.setdefault('filt2','f200w')
    filters.setdefault('filt3', filters['filt2'])
    
    positions.setdefault('ra_col','ra')
    positions.setdefault('dec_col','dec')
    positions.setdefault('ra_cen',0)
    positions.setdefault('dec_cen',0)
    
    region.setdefault('r_in',0)
    region.setdefault('r_out',10)
    region.setdefault('spatial_filter','circle')
    
    extinction.setdefault('Av',0.19)
    extinction.setdefault('Av_',3)
    extinction.setdefault('Av_x',3)
    extinction.setdefault('Av_y',19)
    
    axis_limits.setdefault('xlims', [-0.5, 2.5])
    axis_limits.setdefault('ylims', [18, 28])
    
    isochrone_params.setdefault('label_min', 0)
    isochrone_params.setdefault('label_max', 10)
    isochrone_params.setdefault('met', [0.02])
    isochrone_params.setdefault('age', [7,8,9])
    isochrone_params.setdefault('mask_fact', 1)
    
    plot_settings.setdefault('Av.fontsize',15)
    plot_settings.setdefault('legend.fontsize',15)
    plot_settings.setdefault('lw',3)
    plot_settings.setdefault('s',0.2)
    plot_settings.setdefault('alpha',1)
    plot_settings.setdefault('print_met',False)
    plot_settings.setdefault('legend.ncols',1)
    
    
    error_settings.setdefault('mag_err_cols', [
        f'mag_err_{filters["filt1"].upper()}',
        f'mag_err_{filters["filt2"].upper()}',
        f'mag_err_{filters["filt3"].upper()}',])
    
    error_settings.setdefault('mag_err_lim',0.2)
    error_settings.setdefault('ref_xpos',-0.25)
    
    kde_contours.setdefault('gen_kde',False)
    kde_contours.setdefault('gen_contours',False)
    kde_contours.setdefault('kde_bin',100j)
    kde_contours.setdefault('cmap','jet')
    kde_contours.setdefault('bw', 0.05)
    
    other_settings.setdefault('ab_dist',True)
    other_settings.setdefault('skip_data',False)
    other_settings.setdefault('show_err_model',False)

    # Filter table by magnitude errors
    for col in error_settings['mag_err_cols']:
        tab = tab[tab[col] <= error_settings['mag_err_lim']]

    # Compute angular separation or define square field
    tab['r'] = angular_separation(
        tab[positions['ra_col']] * u.deg,
        tab[positions['dec_col']] * u.deg,
        positions['ra_cen'] * u.deg,
        positions['dec_cen'] * u.deg).to(u.arcsec).value

    if region['spatial_filter']=='circle':
        tab = tab[(tab['r'] >= region['r_in'])
                  & (tab['r'] <= region['r_out'])]
        
    elif region['spatial_filter']=='box':   
        tab = box(tab, positions['ra_col'], positions['dec_col'],
                  positions['ra_cen'], positions['dec_cen'],
                  region['width_in'] / 3600, region['height_in'] / 3600,
                  region['width_out'] / 3600, region['height_out'] / 3600,
                  region['ang'])

    elif region['spatial_filter']=='ellipse':
        tab = ellipse(tab, positions['ra_col'], positions['dec_col'],
                  positions['ra_cen'], positions['dec_cen'],
                  region['ang'], 
                  region['a1'] / 3600,region['b1'] / 3600,
                  region['a2'] / 3600,region['b2'] / 3600)
    elif region['spatial_filter']=='polygon':
        tab = polygon(tab, positions['ra_col'], positions['dec_col'], region['points'])

    # Compute magnitudes and colors
    x = tab[f'mag_vega_{filters["filt1"].upper()}'] - tab[f'mag_vega_{filters["filt2"].upper()}']
    y = tab[f'mag_vega_{filters["filt3"].upper()}']

    x = x.value.astype(float)
    y = y.value.astype(float)

    # Initialize figure and axis if not provided
    if fig is None or ax is None:
        fig, ax = plt.subplots(figsize=(12, 10))

    # Extinction corrections
    AF1 = Av_dict[filters['filt1']] * extinction['Av']
    AF2 = Av_dict[filters['filt2']] * extinction['Av']
    AF3 = Av_dict[filters['filt3']] * extinction['Av']

    # Kernel density estimation or scatter plot
    tick_color = 'black'
    if kde_contours['gen_kde'] and not kde_contours['gen_contours']:
        xx, yy = np.mgrid[
            axis_limits['xlims'][0]:axis_limits['xlims'][1]:kde_contours['kde_bin'],
            axis_limits['ylims'][0]:axis_limits['ylims'][1]:kde_contours['kde_bin']]
        
        positions = np.vstack([xx.ravel(), yy.ravel()])
        values = np.vstack([x, y])

        kernel = gaussian_kde(values, bw_method=kde_contours['bw'])
        f = np.reshape(kernel(positions), xx.shape)
        tick_color='white'
        norm = simple_norm(f.T, 'log', min_percent=10)
        ax.imshow(f.T, cmap=kde_contours['cmap'], extent=(*axis_limits['xlims'], *axis_limits['ylims']),
                  interpolation='nearest', aspect='auto', norm=norm)

    elif kde_contours['gen_contours']:
        ax.scatter(x, y, s=plot_settings['s'], color='black', label='data')
        cmap_custom = LinearSegmentedColormap.from_list("custom_grey_to_white", ["grey", "white"], N=256)
        sb.kdeplot(x=x, y=y, levels=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99],
                   ax=ax, fill=True, cmap=cmap_custom)
        
        sb.kdeplot(x=x, y=y, levels=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99],
                   ax=ax, color='black')

    elif not other_settings['skip_data']:
        ax.scatter(x, y, s=plot_settings['s'], color='black', label='data')
        
    ax.set_xlim(axis_limits['xlims'][0],axis_limits['xlims'][1])
    ax.set_ylim(axis_limits['ylims'][0],axis_limits['ylims'][1])
    
    ax.tick_params(which='both', length=15,direction="in", 
                   bottom=True, top=True,left=True, width = 3,
                   color=tick_color)
    
    ax.tick_params(which='minor', length=8, width = 3, color=tick_color)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_major_locator(AutoLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    
    # Handle isochrones
        
    age_lin = []
    for i in isochrone_params['ages']:
        if i < 6:
            age_lin.append(f'{np.round(10**i,1)} Myr')
        if i >= 6  and i < 9:
            i -= 6
            age_lin.append(f'{np.round(10**i,1)} Myr')
        elif i >= 9:
            i -= 9
            age_lin.append(f'{np.round(10**i,1)} Gyr')
            
    if df_iso is not None:
        df_iso = df_iso[(df_iso['label']>=isochrone_params['label_min'])
                       & (df_iso['label']<=isochrone_params['label_max'])]
                        
        for i,age in enumerate(isochrone_params['ages']):
            t = df_iso[(np.round(df_iso['logAge'],5) == age)]
            for Z in isochrone_params['met']:
                subset = t[t['Zini'] == Z]
                x_iso = subset[f"{filters['filt1'].upper()}mag"] + AF1 - (
                        subset[f"{filters['filt2'].upper()}mag"] + AF2)
                y_iso = subset[f"{filters['filt3'].upper()}mag"] + AF3 + distance_modulus
                       
                mask = (y_iso.values[1:]- y_iso.values[:-1])<=isochrone_params['mask_fact']
                mask = np.array([True] + list(mask))
                mask = np.where(~mask, np.nan, 1)
                
                if len(isochrone_params['met'])>1 or plot_settings['print_met']:
                    label = label=age_lin[i]+ f' {Z}'
                else:
                    label = label=age_lin[i]
                               
                ax.plot(x_iso*mask, y_iso*mask, lw=plot_settings['lw'],
                        label=label,alpha=plot_settings['alpha'])

    # Absolute magnitude
    if other_settings['ab_dist']:
        yticks = ax.get_yticks()
        yticks_n = yticks - distance_modulus - AF3
        
        dy = yticks_n - np.floor(yticks_n)
        ax1 = ax.twinx()  # instantiate a second axes that shares the same x-axis            
        ax1.set_ylabel(r'$M_{' + f"{filters['filt3'].upper()}" + r'}$')  # we already handled the x-label with ax1
        ax1.set_yticks(yticks - dy, np.floor(yticks_n), fontsize=30)
        ax1.yaxis.set_minor_locator(AutoMinorLocator())
        
        ax1.set_xlim(axis_limits['xlims'][0],axis_limits['xlims'][1])
        ax1.set_ylim(axis_limits['ylims'][0],axis_limits['ylims'][1])
        
        ax1.tick_params(which='both', length=15,direction="in",
                        right=True, width = 3, color=tick_color)
        ax1.tick_params(which='minor', length=8, width = 3, color=tick_color)
        
        
        ax1.invert_yaxis()

    # Extinction Vector
    AF1_ = Av_dict[filters['filt1']] * extinction['Av_']
    AF2_ = Av_dict[filters['filt2']] * extinction['Av_']
    AF3_ = Av_dict[filters['filt3']] * extinction['Av_']

    dx = AF1_ - AF2_
    dy = AF3_

    ax.annotate('', xy=(extinction['Av_x'], extinction['Av_y']),
                 xycoords='data',
                 xytext=(extinction['Av_x']+dx, 
                         extinction['Av_y']+dy),
                 textcoords='data',
                 arrowprops=dict(arrowstyle= '<|-',
                                 color=tick_color,
                                 lw=plot_settings['lw'],
                                 ls='-')
               )

    ax.annotate(f"Av = {extinction['Av_']}",
                xy=(extinction['Av_x']-0.1, extinction['Av_y']-0.1)
                ,fontsize=plot_settings['Av.fontsize'],
                color=tick_color)
    
    # Error models
    if not other_settings['skip_data']:
        ref = tab[f"mag_vega_{filters['filt3'].upper()}"]
        ref_new = np.arange(np.ceil(np.nanmin(y)),np.floor(np.nanmax(y))+0.5,0.5)

        mag_err1 = tab[error_settings['mag_err_cols'][0]]
        mag_err2 = tab[error_settings['mag_err_cols'][1]]

        if len(error_settings['mag_err_cols'])>2:
            mag_err3 = tab[error_settings['mag_err_cols'][2]]
        else:
            mag_err3 = mag_err2

        col_err = np.sqrt(mag_err1**2 + mag_err2**2)
        init = models.Exponential1D()
        fit = fitting.LevMarLSQFitter()
        model_col = fit(init,ref,col_err)

        init = models.Exponential1D()
        fit = fitting.LevMarLSQFitter()
        model_mag = fit(init,ref,mag_err3)

        x = error_settings['ref_xpos'] + 0*ref_new
        y = ref_new
        yerr = model_mag(ref_new)
        xerr = model_col(ref_new)
        
        if other_settings['show_err_model']:
            plt.show()
            plt.scatter(ref, mag_err3)
            plt.plot(ref_new,yerr,'--r')
            plt.show()
            plt.scatter(ref, col_err)
            plt.plot(ref_new,xerr,'--r')
            plt.show()
        ax.errorbar(x, y, yerr, xerr ,fmt='o', color = 'red', markersize=0.5, capsize=2) 
    
    # Labels, ticks, and legend
    ax.invert_yaxis()
    ax.set_xlabel(f"{filters['filt1'].upper()} - {filters['filt2'].upper()}")
    ax.set_ylabel(filters['filt3'].upper())
    ax.legend(fontsize=plot_settings['legend.fontsize'], ncols = plot_settings['legend.ncols'])

    fig.tight_layout()
    return fig, ax, tab

# **Simulation Parameters**

In [ ]:
ages = np.arange(6.6,10.01,0.1)
ages_l = ages - 0.05
ages_r = ages + 0.05
delta_t = 10**ages_r - 10**ages_l

SFRs = 5e7/delta_t

In [ ]:
(SFRs*delta_t).sum()

In [ ]:
print(f"{10**ages[0]-0.1} 0 0.002 0.0")
for age, SFR in zip(ages, SFRs):
    print(f"{10**age:0.5f} {SFR:0.5f} 0.002 0.0")
print(f"{10**ages[-1]+0.1} 0 0.002 0.0")

In [ ]:
plt.plot(ages, SFRs)
plt.yscale('log')

In [ ]:
df_cmd_jwst = pd.read_csv("../data/isochrones_master/cmd_jwst_master.csv")

In [ ]:
fs = glob.glob("../TRILEGAL/jwst_iso.dat")
df = []
for fi in  fs:
    with open(fi) as f:
        dat = f.readlines()
    
    data = []
    
    for i,d in enumerate(dat[:]):
        if 'Zini' not in d and 'terminated' not in d:
            data.append([float(i) for i in d.split()])
                
    df_cmd = pd.DataFrame(data,columns=dat[0][2:].split())
    df.append(df_cmd)
df_cmd = pd.concat(df)
df_cmd2 = df_cmd.drop_duplicates(['Mini','logAge','label'])

In [ ]:
fs = glob.glob("../TRILEGAL/jwst_const_SFR.dat")
df = []

for fi in  fs:
    with open(fi) as f:
        dat = f.readlines()
    
    data = []
    
    for i,d in enumerate(dat[:]):
        if 'Z' not in d and 'terminated' not in d:
            data.append([float(i) for i in d.split()])
                
    df_sim = pd.DataFrame(data,columns=dat[0][1:].split())
    df.append(df_sim)
    
df_sim = Table.from_pandas(pd.concat(df))


In [ ]:
model_f115w = models.Exponential1D(1.38212563e-09, 1.48664425e+00)
model_f200w = models.Exponential1D(1.46392219e-09, 1.45003324e+00)

cols =  {}
cols_err = []
for i in df_sim.keys():
    if 'mag' in i and 'F' in i:
        cols[i]  = f'mag_vega_{i[:-3]}'
        cols_err.append(f'mag_err_{i[:-3]}')
        
df_sim.rename_columns(list(cols.keys()), list(cols.values()))

df_sim['mag_err_F115W'] = model_f115w(df_sim['mag_vega_F115W']) 
df_sim['mag_vega_F115W'] = np.random.normal(loc=df_sim['mag_vega_F115W'], scale = df_sim['mag_err_F115W'])

df_sim['mag_err_F200W'] = model_f200w(df_sim['mag_vega_F200W']) 
df_sim['mag_vega_F200W'] = np.random.normal(loc=df_sim['mag_vega_F200W'], scale = df_sim['mag_err_F200W'])

df_sim['ra']  = 24.185745
df_sim['dec'] = 15.772426

In [ ]:
# ages_select =  [6.69897, 6.77815, 6.81291, 6.8451 ,  6.90309,
#        6.92942, 6.95424, 6.97772, 7., 7.04139,  7.09691, 7.13033,7.17609, 7.25527, 7.31175
           
ages_select= [7,  ]
# [   ,  7.26717, 7.27875, 7.29003, , ,]

In [ ]:
ind

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,5))
F200W = []
for age in ages:
    df_temp = df_cmd[(df_cmd['logAge']==age) & 
        (df_cmd['label']>0) & 
        (df_cmd['label']<7) & 
        (df_cmd['logTe']<4.5) &
        (df_cmd['logTe']>0) 
        #(df_cmd['Mini']<35)
        ]
    if age>7.7:
        df_temp = df_temp[df_temp['label']>1]

    x = df_temp['logTe']
    y = df_temp['logL']

    ax[0].plot(x,y)

    x = df_temp['F115Wmag'].values - df_temp['F200Wmag'].values
    y = df_temp['F200Wmag'].values + 29.81
    ind = np.argsort(x)
    ax[1].scatter(x,y)
    col = 0.3
    F200W.append(np.interp(col, x[ind],y[ind]))
    ax[1].scatter(col, np.interp(col,x[ind],y[ind]), marker='x', color='black')
    
ax[0].set_xlabel('logTe')
ax[0].set_ylabel('logL')
ax[0].invert_xaxis()

ax[1].set_xlabel('F115W - F200W')
ax[1].set_ylabel('F200W')
ax[1].invert_yaxis()

In [ ]:
model_f200w = models.Exponential1D(1.46392219e-09, 1.45003324e+00)

In [ ]:
age_flag = [True]
age_c = F200W[0]
for i in range(len(ages)-1):
    if F200W[i+1] - age_c>model_f200w(age_c)*5:
        age_flag.append(True)
        age_c =  F200W[i+1]
    else:
        age_flag.append(False)

In [ ]:
np.round(10**ages[age_flag]/1e6,2)

In [ ]:
len(ages[age_flag])

In [ ]:
tab = Table.read('../TRILEGAL/jwst_M5e7_Z002.fits')
fig, ax = plt.subplots(figsize=(12,10))
ra_cen  = 24.185745
dec_cen = 15.772426

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f200w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

region = {'r_in'    : 0,
          'r_out': 2000,
          'spatial_filter': 'circle'}

extinction = {'Av'  : 0,
              'Av_x': 4,
              'Av_y': 19,
              'Av_' : 2}

axis_limits= {'xlims': [-1, 3], 
              'ylims': [17, 28]}

isochrone_params={'met': [0.02],
                  'label_min': 0,
                  'label_max': 8,
                  'ages': ages[age_flag],
                  'mask_fact' : 5}

error_settings = {'ref_xpos': -0.5,
                  'mag_err_lim':0.2}

plot_settings = {'s':2, 'legend.ncols':5, 'alpha':0.7, 'lw':2}


fig,ax, tab1 = gen_CMD(tab, 
                      df_cmd2,
                      filters, 
                      positions,
                      region,
                      extinction,
                      29.81,
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)

isochrone_params['ages'] = [10.0]
isochrone_params['met'] =  [0.002]

fig,ax, tab1 = gen_CMD(tab, 
                      None,
                      filters, 
                      positions,
                      region,
                      extinction,
                      29.81,
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': True, 'skip_data': True},
                      fig=fig, ax=ax)
l = ax.legend()
#l.remove()

In [ ]:
df_sim.write("../TRILEGAL/jwst_M5e7.fits", overwrite=True)
len(tab)

In [ ]:
from pathlib import Path

In [ ]:
f = Path("../TRILEGAL/jwst_M5e7.fits")

In [ ]:
f.stem

In [ ]:
x = df_sim['logAge'] 
y = df_sim['Z']

plt.hist(y,log=True)

In [ ]:
plt.hist(x,log=True)

In [ ]:
x = 10**df_sim['logAge']/1e9
y = df_sim['Z']

plt.scatter(x,y)

In [ ]:
np.log10(df_sim['m_ini'].sum())